<div style="background: linear-gradient(120deg, #1a3a5c 0%, #2d6a9f 60%, #4a9eda 100%); padding: 28px 36px; border-radius: 14px; display: flex; align-items: center; gap: 28px; box-shadow: 0 4px 18px rgba(0,0,0,0.18);">
    <img src='Figures/iteso.jpg' style="height: 110px; border-radius: 8px; background: white; padding: 6px; flex-shrink: 0; box-shadow: 0 2px 8px rgba(0,0,0,0.2);"/>
    <div style="border-left: 2px solid rgba(255,255,255,0.4); padding-left: 28px;">
        <h1 style="margin: 0 0 8px 0; color: white; font-size: 1.5em; line-height: 1.3;">Maestría en Ciencia de Datos</h1>
        <h3 style="margin: 0 0 8px 0; color: white; font-size: 1.5em; line-height: 1.3;">Ingeniería de Características</h3>
        <h3 style="margin: 0; color: rgba(255,255,255,0.8); font-weight: normal; font-size: 1.05em;">Módulo 1: Extracción de datos de diferentes fuentes</h3>
    </div>
</div>



## Propósito

La extracción es el primer paso para convertir información heterogénea en datos analizables. En este cuaderno se practican técnicas para leer archivos tabulares, texto libre, documentos jerárquicos, datos geoespaciales, páginas web e imágenes.

### Resultados de aprendizaje

Al finalizar podrás:

- elegir una estrategia de lectura según la estructura de la fuente;
- inspeccionar tipos, dimensiones, columnas y valores antes de analizar;
- transformar datos anidados o no estructurados en tablas;
- identificar riesgos de calidad, codificación, rutas y reproducibilidad.

> **Idea central:** extraer datos no significa únicamente abrir un archivo. Significa conservar su significado, documentar los supuestos y producir una representación útil para la siguiente etapa del flujo de ciencia de datos.

```mermaid
flowchart LR
    A[Fuente original] --> B[Lectura]
    B --> C[Inspección]
    C --> D[Limpieza y transformación]
    D --> E[DataFrame o matriz]
    E --> F[Análisis y modelado]
```

## 1. Preparar el entorno y las rutas

Antes de leer una fuente conviene centralizar la carpeta de datos. Así se evita repetir rutas, se facilita mover el cuaderno y se hace más clara la procedencia de cada archivo.

En un proyecto reproducible, la ruta debe ser relativa al proyecto o construirse con `pathlib`. En los ejemplos siguientes se conserva la variable `ruta` para trabajar con los archivos didácticos del curso.


In [ ]:
# Almacenamos la ruta de los archivos en una variable
ruta='Data/'

# 2. Archivos Excel y CSV

Los archivos CSV y Excel representan datos tabulares: cada fila suele corresponder a una observación y cada columna a una variable. Son frecuentes en reportes, exportaciones de sistemas y conjuntos de datos pequeños o medianos.

`pandas` permite cargarlos en un `DataFrame`, una estructura que facilita filtrar, resumir y transformar datos. La lectura correcta depende de detalles como el separador, la codificación, los encabezados y los valores faltantes.

**Checklist de inspección:** después de cargar un archivo revisa `shape`, `columns`, `dtypes`, valores nulos y algunas filas. En el ejemplo se elimina `Unnamed: 0`, una columna índice que suele aparecer cuando un DataFrame se exportó sin `index=False`.


In [ ]:
import pandas as pd

In [ ]:
df_csv = pd.read_csv(ruta+'df_tabla2.csv')
print('Contenido del archivo CSV:')
print(df_csv)

In [ ]:
df_csv

In [ ]:
df_csv.iloc[:,0]

In [ ]:
df_csv = df_csv.drop('Unnamed: 0', axis=1)
df_csv

# 3. Archivos de texto

Los archivos de texto no siempre tienen una estructura explícita. Para extraerlos hay que determinar si sus campos están separados por tabuladores, comas, espacios u otro delimitador, o si cada columna ocupa un ancho fijo.

### De texto a `DataFrame`

La estrategia elegida debe corresponder al formato real. Si el separador cambia entre filas, hay que normalizar la fuente antes de convertirla en tabla; si el archivo contiene texto libre, conviene leerlo como una cadena y aplicar técnicas de procesamiento de lenguaje.


`pd.read_fwf` de pandas se utiliza para leer archivos de texto con columnas de ancho fijo (*fixed-width formatted lines*) y cargarlos en un DataFrame. En este formato, la posición de los caracteres define cada columna, no un separador visible.

Es útil para reportes heredados o archivos generados por sistemas antiguos. Antes de usarlo, confirma que las posiciones de inicio y fin sean consistentes en varias filas.


In [ ]:
# Extracción a partir de texto separado por tabular
pd.read_fwf(ruta+'texto_2.txt',header=None) # No se puede especificar el separador

`read_table` de pandas se utiliza para leer archivos de texto delimitados (por defecto, separados por tabulaciones) y cargarlos en un DataFrame

In [ ]:
pd.read_table(ruta+'texto_2.txt',header=None) # sep='\t'

In [ ]:
# Extracción a partir de texto separado por comas
pd.read_table(ruta+'texto_1.txt',sep=',',header=None)

In [ ]:
pd.read_table(ruta+'texto_1.txt',sep=' ',header=None)

In [ ]:
pd.read_csv(ruta+'texto_2.txt',header=None,sep='\t') # sep=','

In [ ]:
# Conversión de archivo a variable
file=open(ruta+'texto_3.txt')     # Abrir...
texto=file.read()
file.close()                      # ...despues cerrar

In [ ]:
texto

In [ ]:
# Si ocurre un error durante la ejecución, la variable file se cierra siempre:
with open(ruta+'texto_3.txt') as file:
  texto=file.read()

In [ ]:
texto

In [ ]:
# Separamos cada palabra de la variable de texto
texto.split()

In [ ]:
texto.split?

### 3.1 Expresiones regulares

Las **expresiones regulares** (*regex*) describen patrones de texto. Son apropiadas cuando la fuente no tiene una estructura tabular, pero sí reglas repetibles: correos, fechas, identificadores o códigos.

El flujo habitual es: definir el patrón, probarlo con ejemplos representativos, extraer coincidencias y validar los casos que no coincidan. Una regex demasiado general puede capturar datos incorrectos; una demasiado estricta puede omitir variaciones válidas.


Las **expresiones regulares** (regex) son patrones que se utilizan para buscar, extraer o manipular texto de manera flexible y eficiente. Permiten identificar cadenas de texto que cumplen ciertas reglas, como correos electrónicos, números de teléfono, palabras específicas, etc.

##### ¿Cómo se usan en Python?

En Python, se utiliza el módulo `re` para trabajar con expresiones regulares. Algunas funciones comunes son:

- `re.search()`: busca un patrón en cualquier posición y devuelve la primera coincidencia.
- `re.match()`: comprueba la coincidencia desde el inicio de la cadena.
- `re.findall()`: devuelve todas las coincidencias como una lista.
- `re.split()`: divide el texto usando el patrón como separador.
- `re.sub()`: reemplaza coincidencias por otro texto.

##### ¿Cómo definir patrones en regex?

- Los patrones se definen como cadenas de texto; el prefijo `r` evita que Python interprete algunas barras invertidas antes que regex.
- Algunos caracteres especiales:
  - `.`: cualquier carácter excepto salto de línea
  - `\d`: un dígito (0-9)
  - `\w`: un carácter alfanumérico o guion bajo
  - `\s`: un espacio en blanco
  - `*`: cero o más repeticiones
  - `+`: una o más repeticiones
  - `?`: cero o una repetición
  - `^`: inicio de línea
  - `$`: fin de línea
  - `[abc]`: cualquier carácter a, b o c
  - `( )`: agrupación y captura

> **Lectura del ejemplo:** `\w+@\w+\.\w+` busca una estructura simple de correo. Es útil para aprender, pero no pretende validar todas las direcciones permitidas por los estándares de correo.


###### Ejemplo: Correos electrónicos

In [ ]:
import re

texto_correo = "Mi correo es gdesirena@iteso.mx pero anteriormente usaba gdesirena@gmail.com"
patron = r"\w+@\w+\.\w+"  # patrón para un correo electrónico

resultado = re.search(patron, texto_correo)
if resultado:
    print("Correo encontrado:", resultado.group())

In [ ]:
correos = re.findall(patron, texto_correo) #Encuentra todos los correos en texto_correo
correos

In [ ]:
df_correos = pd.DataFrame({'correo': correos})
df_correos

In [ ]:
dominios = re.findall(r'@([\w\.-]+)', texto_correo) #captura el dominio después del @
print('Dominios encontrados:', dominios)

In [ ]:
texto_fechas = 'algunas fechas importantes son 15/09/2023 y 01/01/2024.' 
fechas = re.findall(r'\b\d{2}/\d{2}/\d{4}\b', texto_fechas) #Encuentra todas las fechas
print('Fechas encontradas:', fechas)

###### Ejemplo: Conteo de palabras usando regex

In [ ]:
#re.split?

In [ ]:
L=re.split(r'\W',texto)
L

In [ ]:
# Convertimos la lista a set
S=set(L)
S.discard('')
S

In [ ]:
# Buscamos las palabras no repetidas del set en la variable de texto para poderlas contar
re.findall?

In [ ]:
# En, en

In [ ]:
re.findall('en',texto,flags=re.I)

In [ ]:
d={}
for palabra in S:
    d[palabra]=len(re.findall(palabra,texto,flags=re.I))
d

In [ ]:
df=pd.DataFrame(d.items(),columns=['Palabra','No.'])

In [ ]:
df.head()

https://regex101.com/

# 4. Archivos Excel

Excel puede contener varias hojas, fórmulas, índices y celdas de presentación. Al extraer datos, distingue la hoja que contiene la tabla de las filas de título o notas que sirven para lectura humana.

`pd.read_excel` es conveniente para leer directamente una hoja. `ExcelFile` resulta útil cuando se desea inspeccionar o reutilizar un libro con varias hojas sin abrirlo repetidamente.

> **Buenas prácticas:** registra el nombre de la hoja, comprueba los encabezados y convierte explícitamente fechas y números cuando Excel los haya interpretado de forma ambigua.


In [ ]:
# A partir de la función
pd.read_excel(ruta+'API_SI.POV.DDAY_DS2_en_excel_v2_1930012.xls')

In [ ]:
# Importamos la clase ExcelFile
from pandas import ExcelFile

In [ ]:
# A partir de la clase
obj=ExcelFile(ruta+'API_SI.POV.DDAY_DS2_en_excel_v2_1930012.xls')
obj.parse() # Importa la primera página

# 5. Archivos JSON

JSON representa datos mediante pares `clave: valor`, listas y objetos anidados. Es muy habitual en APIs, configuraciones y registros de aplicaciones porque conserva mejor la estructura que una tabla plana.

El proceso tiene dos etapas: `json.load` o `json.loads` convierte el contenido a diccionarios y listas de Python; después, `json_normalize` ayuda a convertir objetos repetidos o anidados en columnas de un DataFrame. No todos los JSON tienen la misma forma, por lo que primero conviene inspeccionar sus claves y niveles.


El formato JSON (JavaScript Object Notation) es ampliamente utilizado para el intercambio de datos, especialmente en aplicaciones web y APIs. Python incluye la librería estándar `json` para leer y manipular archivos JSON.

In [ ]:
import json

json_data = '{"personas": [{"nombre": "Ana", "edad": 23}, {"nombre": "Luis", "edad": 31}]}'
data = json.loads(json_data)
print('Personas extraídas del archivo JSON:')
for persona in data['personas']:
    print(f'Nombre: {persona["nombre"]}, Edad: {persona["edad"]}')

In [ ]:
from pandas import json_normalize

with open('Data/data.json') as jsonfile:
    jsondata = json.load(jsonfile)
    
df_json = json_normalize(jsondata['data'])
df_json

In [ ]:
df_json['categories'][0]


# 6. Archivos XML

XML organiza la información como un árbol de elementos. Cada nodo puede tener una etiqueta, atributos, texto y nodos hijos. Esta jerarquía permite representar relaciones complejas, aunque requiere recorrer el árbol para llegar a los valores.

En los ejemplos se usan dos enfoques: `find`/`findall` para rutas conocidas y recorridos anidados cuando se necesita explorar la estructura. Al convertir XML a tabla hay que decidir qué nodo representa una fila y cómo tratar los elementos que faltan o se repiten.


El formato XML es común para el intercambio de datos estructurados. Python ofrece la librería estándar `xml.etree.ElementTree` para analizar y extraer información de archivos XML.

In [ ]:
import xml.etree.ElementTree as ET

In [ ]:
xml_data = '''
<personas>
  <persona>
    <nombre>Ana</nombre>
    <edad>23</edad>
  </persona>
  <persona>
    <nombre>Luis</nombre>
    <edad>31</edad>
  </persona>
</personas>
'''

In [ ]:

root = ET.fromstring(xml_data)
print('Personas extraídas del archivo XML:')
for persona in root.findall('persona'):
    nombre = persona.find('nombre').text
    edad = persona.find('edad').text
    print(f'Nombre: {nombre}, Edad: {edad}')

###### Ejemplo con tabla_1.xml


In [ ]:
archivo_1=ET.parse(ruta+'tabla_1.xml')
raiz=archivo_1.getroot()

In [ ]:
raiz

In [ ]:
for nodo in raiz:
    print(nodo.attrib,nodo.text,nodo.tag)
    for sn in nodo:
        print(sn.attrib,sn.text,sn.tag)

In [ ]:
archivo_2=ET.parse(ruta+'tabla_2.xml')
root=archivo_2.getroot()
for nodo in root:
    print(nodo.tag,nodo.attrib,nodo.text)

In [ ]:
for nodo in root:
    for subn in nodo:
        print(subn.tag,subn.attrib,subn.text)

In [ ]:
#Extraer los datos de tabla_1.xml
d={}
for nodo in raiz:
    d[nodo.tag]=[]
for nodo in raiz:
    d[nodo.tag].append(nodo.attrib['name'])
for nodo in raiz:
    for sn in nodo:
        d[sn.tag]=[]
for nodo in raiz:
    for sn in nodo:
        d[sn.tag].append(sn.text)
d

In [ ]:
pd.DataFrame(d)

```python
df_1 = pd.DataFrame(columns = columnas)
for nodo in raiz:
  L = []
  L.append(nodo.attrib['name'])
  for sn in nodo:
    L.append(sn.text)
  df_1 = df_1.append(pd.DataFrame([L], columns=columnas), ignore_index=True)
```

In [ ]:
archivo=ET.parse(ruta+'tabla_2.xml')
raiz=archivo.getroot()

In [ ]:
L=[]
for n in raiz.findall('documents/document'):
    d={}
    d[n.tag]=n.text
    for k,v in n.attrib.items():
        d[k]=v
    L.append(d)
pd.DataFrame(L)

---

In [ ]:
archivo='IFC-Subscriptions-and-Voting-Power-of-Member-Count.xml'
file=ET.parse(ruta+archivo)
root=file.getroot()

for nodo in root:
    for snodo in nodo:
        print(snodo.tag,snodo.attrib,snodo.text)
        for ssnodo in snodo:
            print(ssnodo.tag,ssnodo.attrib,ssnodo.text)

---

# 7. Archivos SHP

Los archivos Shapefile (`.shp`) almacenan geometrías vectoriales, como puntos, líneas o polígonos, junto con atributos descriptivos. En realidad forman un conjunto de archivos relacionados: `.shp` guarda la geometría, `.shx` el índice espacial y `.dbf` los atributos; el `.prj` describe el sistema de coordenadas cuando está presente.

`geopandas` extiende `pandas` con una columna `geometry` y un sistema de referencia espacial (`crs`). Antes de calcular distancias o superponer capas, verifica que las capas estén en un CRS adecuado: los grados de latitud/longitud no son unidades lineales.


La librería `geopandas` permite leer y manipular estos archivos de forma parecida a un DataFrame, pero manteniendo la geometría. La visualización temática del ejemplo colorea cada geometría usando una columna de atributos; esto ayuda a comprobar que la tabla y el mapa se alinean.

> **Nota de instalación:** las celdas de instalación se conservan como referencia. En un entorno compartido es preferible instalar dependencias una sola vez y fijar sus versiones en `environment.yml` o `requirements.txt`.


In [ ]:
!pip install geopandas

In [ ]:
#!pip install geopandas

In [ ]:
# %conda !pip !conda
%pip install geopandas

In [ ]:
import geopandas as gpd

In [ ]:
g_df = gpd.read_file('COVID_INDIA_POC-shp/COVID_INDIA_POC.shp')
g_df

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(12, 8))
g_df.plot(
    ax=ax,
    column=g_df.columns[0],  # Cambia por la columna que quieras destacar
    cmap='viridis',
    edgecolor='black',
    legend=True
)
ax.set_title('COVID_INDIA_POC', fontsize=12)
ax.axis('off')
plt.show()

# 8. Archivos HTML y web scraping

HTML describe la estructura de una página mediante etiquetas y atributos. La extracción puede hacerse desde un archivo local o desde una URL, pero en ambos casos conviene separar tres tareas: descargar o abrir el documento, analizar su árbol y seleccionar los elementos que contienen los datos.

`BeautifulSoup` facilita la navegación del HTML. En sitios reales, los selectores pueden cambiar, existir contenido generado con JavaScript o aplicarse restricciones de uso; por eso hay que revisar los términos del sitio, limitar las solicitudes y comprobar que los datos extraídos sean completos.


In [ ]:
# Leer un archivo HTML local
with open(ruta+'ejemplo.html', 'r', encoding='utf-8') as file:
    html_content = file.read()
print(html_content[:500])  # Muestra los primeros 500 caracteres

In [ ]:
#!pip install beautifulsoup4 #Instalar BeautifulSoup si es necesario

In [ ]:
# Analizar HTML con BeautifulSoup
from bs4 import BeautifulSoup
soup = BeautifulSoup(html_content, 'html.parser')

# Extraer el título de la página
titulo = soup.title.string
print('Título de la página:', titulo)

In [ ]:
# Extraer los enlaces de la página
enlaces = soup.find_all('a')
for enlace in enlaces:
    print(enlace.get('href'))

In [ ]:
#!pip install requests # Instalar requests si es necesario
import requests

url = 'https://www.python.org/'
response = requests.get(url)
web_html = response.text

# Analizar el HTML descargado
soup_web = BeautifulSoup(web_html, 'html.parser')


In [ ]:
print(soup_web.title)

###### Ejemplo sencillo webscraping 

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
 
url = "https://codedamn-classrooms.github.io/webscraper-python-codedamn-classroom-website/"
response = requests.get(url)
soup = BeautifulSoup(response.text, "html.parser")
type(soup)

In [ ]:
#Obtener un dataframe con la información de los productos
products = []

for price_tag in soup.find_all("h4", string=lambda s: s and s.strip().startswith("$")):
    # El nombre del producto es la cadena que sigue después del tag <a> después del precio
    product_link = price_tag.find_next("a")
    if not product_link:
        continue
    product_name = product_link.text.strip()
    product_url = product_link["href"]
    # La descripción es el siguien tag <p> después de la liga del producto
    description_tag = product_link.find_next("p")
    description = description_tag.text.strip() if description_tag else ""
    # Los numero de reviews van después de <div> 
    reviews_tag = product_link.find_next(string=lambda s: s and "review" in s)
    try:
        reviews = int(reviews_tag.strip().split()[0])
    except Exception:
        reviews = None
    # Precio
    try:
        price = float(price_tag.text.strip().replace("$", ""))
    except Exception:
        price = None
 
    products.append({
        "Product Name": product_name,
        "Price": price,
        "Description": description,
        "Reviews": reviews,
        "Product URL": product_url
    })
 
df = pd.DataFrame(products)
df

# 9. Archivos de imagen

Una imagen digital puede interpretarse como una matriz de píxeles. En una imagen RGB, la forma suele ser `(alto, ancho, 3)`: el último eje contiene los canales rojo, verde y azul. En imágenes con transparencia puede aparecer un cuarto canal alfa.

Leer una imagen como arreglo permite crear características numéricas, por ejemplo promedios de color, histogramas, bordes o texturas. Antes de modelar conviene revisar el rango de valores, el tipo de dato, la orientación y si todas las imágenes tienen la misma resolución.


Las imágenes RGB almacenan información de color en tres canales: Rojo (R), Verde (G) y Azul (B). Para leer y manipular imágenes en Python, se pueden usar las librerías `Pillow` (PIL), `matplotlib` y `numpy`. Esto permite acceder a los valores de los píxeles y realizar análisis o transformaciones.

En las celdas siguientes se observa la imagen completa, un canal individual y una versión en escala de grises. La conversión mediante el promedio es didáctica; para aplicaciones sensibles a la percepción humana suele ser preferible una transformación ponderada como la de luminancia.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
I=plt.imread(ruta+'imagen.bmp')
I.shape

In [ ]:
type(I)

In [ ]:
plt.imshow(I)

In [ ]:
I[0,0,0] # pixel (0,0) de la matriz roja

In [ ]:
plt.imshow(I[:,:,0],cmap='gray') # Matriz roja

In [ ]:
G=I.mean(axis=2)
G.shape

In [ ]:
plt.imshow(G)

In [ ]:
G[0,0]

# 10. Aplicaciones en Ciencias de Datos

La fuente determina qué información puede conservarse y qué transformaciones serán necesarias. La siguiente guía conecta cada formato con tareas frecuentes de análisis y ayuda a elegir una primera estrategia de extracción.

```mermaid
flowchart TD
    Q{¿Cómo está organizada la fuente?}
    Q -->|Filas y columnas| T[CSV o Excel]
    Q -->|Texto libre| X[Texto y regex]
    Q -->|Objetos anidados| J[JSON o XML]
    Q -->|Geometría y atributos| S[Shapefile]
    Q -->|Etiquetas de una página| H[HTML]
    Q -->|Píxeles| I[Imagen]
    T --> R[DataFrame]
    X --> R
    J --> R
    S --> R
    H --> R
    I --> M[Matriz o características]
```


## Archivos de texto

Son adecuados cuando la señal principal está en el lenguaje. Después de extraer el contenido, una canalización típica incluye normalización, tokenización, eliminación o conservación consciente de palabras vacías y generación de características. Las regex funcionan bien para patrones concretos, pero no sustituyen un análisis lingüístico completo.

> **Aplicaciones:** análisis de sentimientos, extracción de palabras clave, procesamiento de lenguaje natural y análisis de logs para detectar patrones o anomalías.


### Archivos Excel y CSV

Son una buena entrada para análisis exploratorio y preparación de datos porque ya expresan observaciones en filas y variables en columnas. La extracción debe ir acompañada de validaciones de tipos, duplicados, valores faltantes y unidades.

> **Aplicaciones:** agrupar ventas por producto, preparar características para modelos de machine learning y generar reportes mensuales de ingresos.


### Archivos de imágenes RGB

La imagen se convierte en variables numéricas antes de entrenar un modelo. Además de los canales de color, pueden extraerse histogramas, bordes, texturas o representaciones aprendidas por redes convolucionales.

> **Aplicaciones:** clasificar dígitos escritos a mano, detectar regiones anómalas en imágenes médicas y calcular histogramas de color.


### Archivos XML

> **Integración de datos de sistemas empresariales:**
  - Extraer información de clientes de un archivo XML exportado de un ERP.
    
> **Procesamiento de datos de sensores o dispositivos IoT:**
  - Leer registros de temperatura almacenados en XML.
    
> **Análisis de datos de publicaciones científicas:**
  - Obtener títulos y autores de artículos en formato XML

### Archivos JSON

> **Consumo de APIs web:**
  - Obtener y analizar tweets desde la API de Twitter.
    
> **Almacenamiento y análisis de logs:**
  - Procesar registros de acceso de una aplicación web.
    
> **Análisis de datos de aplicaciones móviles:**
  - Leer resultados de encuestas exportadas en JSON.

### Archivos Shapefile (SHP)

> **Análisis geoespacial:**
  - Calcular la distancia entre puntos de interés en una ciudad.
    
> **Estudios ambientales y urbanos:**
  - Analizar la distribución de áreas verdes en una zona urbana.

> **Modelado de redes y transporte:**
  - Determinar rutas óptimas entre dos ubicaciones.

## Archivos HTML:
> **Web scraping:** Extraer datos estructurados de páginas web para análisis posterior.

> **Construcción de datasets:** Recolectar información de múltiples páginas HTML para crear conjuntos de datos.

> **Análisis de enlaces:** Estudiar la estructura de enlaces en sitios web para análisis de redes.

> **Extracción de tablas:** Obtener datos tabulares de páginas HTML para análisis estadístico.